# 05 - Full HDFS dataset

Parse the full HDFS_v1 log with Drain3 and build one event sequence and one count vector per block.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.features.hdfs_features import build_hdfs

## Parse and build features

This reads `HDFS.log` line by line and saves both tables to `data/processed/`.

In [2]:
sequences, counts, stats = build_hdfs()

print(f"Total lines:      {stats['lines']:,}")
print(f"Unparsed lines:   {stats['unparsed_lines']:,}")
print(f"Templates:        {stats['templates']}")
print(f"Blocks:           {stats['blocks']:,}")
print(f"Anomalous blocks: {stats['anomalous_blocks']:,} ({stats['anomalous_blocks'] / stats['blocks']:.2%})")
print(f"Total time:       {stats['total_seconds'] / 60:.1f} min")

Parsing HDFS.log: 0 lines [00:00, ? lines/s]

Parsing HDFS.log: 1889679 lines [00:30, 62989.27 lines/s]

Parsing HDFS.log: 1889679 lines [00:41, 62989.27 lines/s]

Parsing HDFS.log: 3679558 lines [01:00, 61032.42 lines/s]

Parsing HDFS.log: 3679558 lines [01:12, 61032.42 lines/s]

Parsing HDFS.log: 5467336 lines [01:30, 60374.95 lines/s]

... (4 outputs left out)

Parsing HDFS.log: 9030793 lines [02:43, 59746.73 lines/s]

Parsing HDFS.log: 10780984 lines [03:00, 59268.34 lines/s]

Parsing HDFS.log: 11175629 lines [03:06, 59833.24 lines/s]

Total lines:      11,175,629
Unparsed lines:   0
Templates:        45
Blocks:           575,061
Anomalous blocks: 16,838 (2.93%)
Total time:       3.7 min


## Example sequences

In [3]:
# two normal blocks and one anomaly, first 30 events each
examples = pd.concat([sequences[sequences["Label"] == 0].head(2), sequences[sequences["Label"] == 1].head(1)])
for row in examples.itertuples():
    label = "anomaly" if row.Label else "normal"
    seq = " ".join(row.EventSequence[:30]) + (" ..." if len(row.EventSequence) > 30 else "")
    print(f"{row.BlockId} ({label}, {len(row.EventSequence)} events): {seq}")

blk_-1608999687919862906 (normal, 269 events): E1 E2 E1 E1 E3 E3 E4 E4 E3 E4 E5 E5 E5 E6 E1 E7 E6 E1 E8 E9 E5 E5 E10 E9 E6 E6 E1 E1 E7 E8 ...
blk_7503483334202473044 (normal, 22 events): E1 E1 E2 E1 E3 E4 E3 E4 E3 E4 E5 E5 E5 E10 E13 E13 E32 E32 E32 E17 E17 E17
blk_-3544583377289625738 (anomaly, 223 events): E1 E2 E1 E1 E3 E4 E3 E4 E3 E4 E10 E5 E5 E5 E10 E10 E10 E10 E10 E10 E10 E10 E10 E10 E10 E10 E10 E10 E10 E10 ...
